In [ ]:
import sys
sys.path.append('${TDL_ROOT_DIR}/John/MNIST_Jan14')
from Trainer import Trainer

In [ ]:
Trainer.root = '${TDL_ROOT_DIR}/John/MNIST_Jan14'
Trainer.results_root = '${TDL_ROOT_DIR}/John/MNIST_Jan14'

In [ ]:
from Trainer import Trainer

# Only patch once
if not hasattr(Trainer, "_original_init"):
    Trainer._original_init = Trainer.__init__

def _patched_init(self, *args, **kwargs):
    if "root" not in kwargs or kwargs["root"] is None:
        kwargs["root"] = "${TDL_ROOT_DIR}/John/MNIST_Jan14"
    Trainer._original_init(self, *args, **kwargs)

Trainer.__init__ = _patched_init


In [ ]:
import dill
import os
os.getcwd()
with open("256x8_leaky/ripser_class_0_norm/model_0.dill", "rb") as f:
    obj = dill.load(f)

print(type(obj))
print(len(obj))
print(type(obj[0]))
print(len(obj[0]))
print(obj[0][:5])
entry = obj[0][0]
print(type(entry))
print(entry)

In [ ]:
import dill

with open("512_32_relu/AGG_TSS/model_0.dill", "rb") as f:
    d = dill.load(f)

print("num layers:", len(d))
print("num dims:", len(d[0]))
for ell in range(len(d)):
    for dim in range(len(d[ell])):
        print(ell, dim, d[ell][dim].shape)


In [ ]:
data = Trainer.all_bootstrap_stats(filename="test",studies = ["512_32_leaky","512_32_relu"], dataset="MNIST", dir_name="ripser_class_0_norm")

In [ ]:
from Trainer import Trainer

_old_init = Trainer.__init__

def _patched_init(self, *args, **kwargs):
    if "root" not in kwargs or kwargs["root"] is None:
        kwargs["root"] = "${TDL_ROOT_DIR}/John/MNIST_Jan14"
    _old_init(self, *args, **kwargs)

Trainer.__init__ = _patched_init


In [ ]:
t = Trainer(
    dataset="MNIST",
    hidden_dims=[30]*8,
    act_fn=None,
    study_name="test"
)

print("root:", t.root)


In [ ]:
from Trainer import Trainer
import numpy as np

_orig = Trainer._betti_at_eta_one_dim

def safe_betti(diagram_dim, eta):
    arr = np.asarray(diagram_dim)

    # 🔒 Enforce (N, 2)
    if arr.ndim == 1 and arr.size == 2:
        arr = arr.reshape(1, 2)

    if arr.ndim != 2 or arr.shape[1] != 2:
        raise ValueError(
            f"Invalid persistence diagram shape inside Trainer: {arr.shape}"
        )

    births = arr[:, 0]
    deaths = arr[:, 1]
    return np.sum((births <= eta) & (eta < deaths))

Trainer._betti_at_eta_one_dim = staticmethod(safe_betti)


In [ ]:
import json
from Trainer import Trainer

# Save original method
if not hasattr(Trainer, "_original_all_bootstrap_stats"):
    Trainer._original_all_bootstrap_stats = Trainer.all_bootstrap_stats

def patched_all_bootstrap_stats(*args, **kwargs):
    out = Trainer._original_all_bootstrap_stats(*args, save=False, **kwargs)

    filename = kwargs.get("filename", "all_models")
    dataset = kwargs.get("dataset", "MNIST")
    p_or_mean = "mean"
    alpha = kwargs.get("alpha", 0.05)

    new_path = (
        f"${TDL_ROOT_DIR}/John/MNIST_Jan14/"
        f"betti_data/{filename}_all_stats_{p_or_mean}_alpha{alpha}.json"
    )

    with open(new_path, "w") as f:
        json.dump(out, f, indent=2)

    print(f"✅ Saved bootstrap stats to:\n{new_path}")
    return out

Trainer.all_bootstrap_stats = patched_all_bootstrap_stats


In [ ]:
data = Trainer.all_bootstrap_stats(
    filename="all_models",
    studies=[
        
        "512_32_leaky", "512_32_relu", "512_32_bottleneck_relu"
    ],
    dataset="MNIST",
    dir_name="AGG_TSS"
)


In [ ]:
Trainer.get_tsc(
    data,  # the output of get_bootstrap_data above
    to_calculate=[r'512'],  # regexes for which models to show in the graph.
                            # TSS would still be computed with respect to all relu and tanh models, as specified above
    title=r'All $512 to $32 models',  # set a title for the graph. Insert latex by using raw string and dollar signs
    legend=['ReLU', 'Leaky'],  # What names to put in the legend of the graph.
                              # First time around, run with legend=None to see which order it puts the models in, and order the legend accordingly
    legend_title='Activation Function',  # Change accordingly (could be just 'Architecture' for example)
    plot=True,
    save=False,  # Change to true to save
    filename='512_32_TSS'  # Filename for saved plot. Change accordingly
)

In [ ]:
Trainer.get_tsc(
    data,
    to_calculate=[r'512'],
    title=r'All $512 \to 32 models',
    legend=None,          
    legend_title='Activation Function',
    plot=True,
    save=False,
    filename='512_32_TSS'
)


In [ ]:
import matplotlib.pyplot as plt

plt.savefig(
    "${TDL_ROOT_DIR}/John/MNIST_Jan14/figures/512_32_TSS.png",
    dpi=300,
    bbox_inches="tight"
)
